In [9]:
#%matplotlib notebook
#%matplotlib widget
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
import os
singlecol = 3.5
plt.rcParams.update({'font.size': 8})
#plt.rcParams.update({'font.sans-serif': 'Helvetica'})
#plt.rcParams['font.family'] = 'dejavu'
plt.rcParams.update({'lines.linewidth':0.5})
plt.rcParams.update({'axes.linewidth':0.5})
plt.rcParams.update({'lines.markersize':2})
plt.rcParams.update({'savefig.dpi':200})
plt.rcParams.update({'legend.fontsize': 7})

In [11]:
import py4vasp                       
mycalc = py4vasp.Calculation.from_path( "." )
mycalc.energy[:].plot( "TOTEN, ETOTAL" )

FileAccessError: Error when reading from /W/yy276937/Projects/AIMD/Na16/vaspout.h5. Please check whether the file format is correct and you have the permissions to read it.

In [ ]:
mycalc.energy[:].plot("TEIN")

In [ ]:
! cat ./REPORT | grep "mc> LV" > ./monitored_cell_volume.dat
from py4vasp import plot
import numpy as np

volume = np.loadtxt("./monitored_cell_volume.dat", usecols=2)

# plot the volume
plot(np.arange(len(volume))+1, volume, "relaxation", xlabel="Number of MD steps", ylabel="Volume (Å³)")

In [ ]:
PCF = mycalc.pair_correlation[:].to_dict()

In [ ]:
plt.figure(figsize = (6,2), dpi=200)
label_list=['C~N','C~Fe','Mn~N','Mn~Fe','N~Na']
for i in range(len(label_list)):
    plt.plot(PCF['distances'], PCF[label_list[i]][0], label=label_list[i])
plt.xlabel('distance (Å)')
plt.yticks([])
plt.legend()
plt.title('Pair Correlation Function')

In [ ]:
! grep ERR ML_LOGFILE > err.dat
t1, ERR = np.loadtxt("./err.dat", usecols=[1,3], unpack=True)

! grep BEEF ML_LOGFILE > beef.dat
t2, BEEF, CTIFOR = np.loadtxt("./beef.dat", usecols=[1,3,5], unpack=True)

! grep "STATUS" ML_LOGFILE|grep -E 'learning|critical'|grep -v "#" >status.dat
when_train = np.loadtxt("./status.dat", usecols=1)

! grep mag OSZICAR > cfg.dat
when_FP = np.loadtxt("./cfg.dat", usecols=0)

In [ ]:
#plt.figure(figsize = (6,2), dpi=200)
fig, ax1 = plt.subplots(figsize = (6,2), dpi=200)
ax1.plot(t1, ERR, label='RMES')
ax1.plot(t2, BEEF, label='Bayesian error')
ax1.plot(t2, CTIFOR, label='Threshold')
for i in range(when_FP.shape[0]):
    ax1.scatter(when_FP[i], BEEF[int(when_FP[i])-1], color='r', s=1)
for i in range(when_train.shape[0]):
    ax1.scatter(when_train[i], BEEF[int(when_train[i])-1], color='blue', s=1)
ax1.legend()
ax1.set_xlabel("Time step")
ax1.set_ylabel("Error in force (eV/Å)")

ax2 = ax1.twinx()
ax2.plot(when_FP, range(when_FP.shape[0]), '--', label='FP calculation', color='red')
ax2.plot(when_train, range(when_train.shape[0]), '--', label='Train MLFF', color='black')
ax2.set_ylabel('How many times')
ax2.legend()

In [ ]:
Force = mycalc.force[:].to_dict()
Force

In [ ]:
Force['structure']['lattice_vectors'][when_FP].shape

In [ ]:
Force['structure']['positions'].shape

In [ ]:
Force['structure']['elements']

In [ ]:
when_FP

In [ ]:
# File paths
ml_abn_format_path = './ML_ABN_format.txt'
train_cfg_format_path = './train.cfg_format'

# Reading ML_ABN_format.txt
data = read_ml_abn_format(ml_abn_format_path)

# Writing to train.cfg_format
write_train_cfg_format(data, train_cfg_format_path)

In [ ]:
def extract_atom_counts(file_path):
    atom_counts = {}
    inside_atom_section = False

    with open(file_path, 'r') as file:
        for line in file:
            line = line.strip()
            if line.startswith("Atom types and atom numbers"):
                inside_atom_section = True
                continue
            if inside_atom_section:
                if line.startswith("="):
                    break
                parts = line.split()
                if len(parts) == 2:
                    atom, count = parts
                    atom_counts[atom] = int(count)
    
    return atom_counts

In [ ]:
def extract_lattice_vectors(file_path):
    lattice_vectors = []
    inside_lattice_section = False

    with open(file_path, 'r') as file:
        for line in file:
            line = line.strip()
            if line.startswith("Primitive lattice vectors (ang.)"):
                inside_lattice_section = True
                continue
            if inside_lattice_section:
                if line.startswith("="):
                    break
                parts = line.split()
                if len(parts) == 3:
                    vector = list(map(float, parts))
                    lattice_vectors.append(vector)
    
    return lattice_vectors

In [ ]:
def extract_atomic_positions(file_path):
    atomic_positions = []
    inside_positions_section = False

    with open(file_path, 'r') as file:
        for line in file:
            line = line.strip()
            if line.startswith("Atomic positions (ang.)"):
                inside_positions_section = True
                continue
            if inside_positions_section:
                if line.startswith("="):
                    break
                parts = line.split()
                if len(parts) == 3:
                    position = list(map(float, parts))
                    atomic_positions.append(position)
    
    return atomic_positions

In [ ]:
def extract_forces(file_path):
    force = []
    inside_force_section = False

    with open(file_path, 'r') as file:
        for line in file:
            line = line.strip()
            if line.startswith("Forces (eV ang.^-1)"):
                inside_force_section = True
                continue
            if inside_force_section:
                if line.startswith("="):
                    break
                parts = line.split()
                if len(parts) == 3:
                    position = list(map(float, parts))
                    force.append(position)
    
    return force

In [ ]:
def is_float(value):
    try:
        float(value)
        return True
    except ValueError:
        return False

def extract_stress(file_path):
    stress = []
    inside_stress_section = False

    with open(file_path, 'r') as file:
        for line in file:
            line = line.strip()
            if line.startswith("Stress (kbar)"):
                inside_stress_section = True
                continue
            if inside_stress_section:
                if line.startswith("="):
                    break
                parts = line.split()
                if len(parts) == 3 and is_float(parts[0]):
                    position = list(map(float, parts))
                    stress.append(position)
    
    return stress

In [ ]:
def extract_total_energy(file_path):
    inside_energy_section = False
    energy = []

    with open(file_path, 'r') as file:
        for line in file:
            line = line.strip()
            if line.startswith("Total energy (eV)"):
                inside_energy_section = True
                continue
            if inside_energy_section:
                if line.startswith("="):
                    break
                parts = line.split()
                if len(parts) == 1 and is_float(parts[0]):
                    position = list(map(float, parts))
                    energy.append(position)
    
    return energy

In [ ]:
import os

def extract_and_save_sections(file_path, output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    sections = {}
    current_section = None
    current_content = []

    with open(file_path, 'r') as file:
        for line in file:
            line = line.strip()
            if line.startswith("Configuration num."):
                if current_section is not None:
                    config_num = current_section.split()[-1]
                    file_name = f"ML_ABN_{config_num}.txt"
                    file_path = os.path.join(output_dir, file_name)
                    with open(file_path, 'w') as section_file:
                        section_file.write("\n".join(current_content))
                    sections[config_num] = file_path
                    current_content = []
                current_section = line
                continue
            if current_section is not None:
                current_content.append(line)
        
        if current_section is not None:
            config_num = current_section.split()[-1]
            file_name = f"ML_ABN_{config_num}.txt"
            file_path = os.path.join(output_dir, file_name)
            with open(file_path, 'w') as section_file:
                section_file.write("\n".join(current_content))
            sections[config_num] = file_path

    return sections

In [ ]:
def write_train_cfg(sections, output_dir):
    atom_type_mapping = {'C': 0, 'Mn': 1, 'Fe': 2, 'N': 3, 'Na': 4}

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    for config_num, file_path in sections.items():
        atom_counts  = extract_atom_counts(file_path)
        lattice_vectors = extract_lattice_vectors(file_path)
        atomic_positions = extract_atomic_positions(file_path)
        forces = np.array(extract_forces(file_path))
        total_energy = extract_total_energy(file_path)[0][0]
        stress_tensor = extract_stress(file_path)

        atom_data = []
        atom_type_list = []
        for atom_type, count in atom_counts.items():
            atom_type_list.extend([atom_type_map[atom_type]] * count)
            
        file_name = f"train.cfg_{config_num}.txt"
        cfg_file_path = os.path.join(output_dir, file_name)
        
        with open(cfg_file_path, 'w') as file:
            file.write("BEGIN_CFG\n")
            file.write(f" Size\n")
            file.write(f"    {len(atomic_positions)}\n")
            file.write(" Supercell\n")
            for vec in lattice_vectors:
                file.write(f"    {'    '.join(f'{v:15.6f}' for v in vec)}\n")
            file.write(" AtomData:  id type       cartes_x      cartes_y      cartes_z           fx          fy          fz\n")
            for i, (pos, force) in enumerate(zip(atomic_positions, forces)):
                atom_type = atom_type_list[i]
                file.write(f"    {i+1:<3} {atom_type:<4} {'    '.join(f'{p:15.6f}' for p in pos)}    {'    '.join(f'{f:15.6f}' for f in force)}\n")
            file.write(f" Energy\n")
            file.write(f"    {total_energy:15.6f}\n")
            if stress_tensor:
                file.write(" PlusStress:  xx          yy          zz          yz          xz          xy\n")
                for tensor in stress_tensor:
                    file.write(f"    {'    '.join(f'{t:15.5f}' for t in tensor)}")
            file.write(" Feature   EFS_by       VASP\n")
            file.write("END_CFG\n")
            file.write("\n")

In [ ]:
file_path = 'ML_ABN'
output_dir = 'ML_ABN_2_cfg/ML_ABN'
sections = extract_and_save_sections(file_path, output_dir)

In [ ]:
# Example usage:
cfg_output_dir = 'ML_ABN_2_cfg/train.cfg'
write_train_cfg(sections, cfg_output_dir)

print(f"train.cfg files have been written to directory: {cfg_output_dir}")

In [ ]:
# Example usage:
file_path = './ML_ABN_format.txt'
#atom_counts, lattice_vectors, atomic_positions, forces, stress, energy= extract_structure(file_path)
atom_counts = extract_atomic_positions(file_path)
lattice_vectors =  np.array(extract_lattice_vectors(file_path))
atomic_positions =  np.array(extract_atomic_positions(file_path))
forces = np.array(extract_forces(file_path))
stress =  np.array(extract_stress(file_path))
total_energy = extract_total_energy(file_path)

In [ ]:
total_energy = extract_total_energy(file_path)
total_energy[0][0]

In [ ]:
file_path = './ML_ABN'
output_dir = 'ML_ABN_2_cfg'

sections = extract_sections(file_path)
write_sections_to_files(sections, output_dir)

In [ ]:
atom_list = ['C', 'Mn', 'Fe', 'N', 'Na']

In [ ]:
atom_counts = extract_atom_counts(file_path)
for i in range(len(atom_counts)):
    print(i)
    atom_numbers = [] 
    atom_numbers.append(atom_counts[atom_list[i]])
    
sum(atom_numbers)

In [ ]:
atom_numbers = sum(atom_counts.values())
atom_numbers

In [ ]:
atom_counts